# EE 451: Communications Systems
## Lesson 12 — FSK Demodulation & Applications

### Learning Objectives
By the end of this lesson, you will be able to:
- Design and analyze coherent FSK detectors using correlators
- Explain non-coherent FSK demodulation using envelope detection
- Describe frequency discriminator operation for FM/FSK demodulation
- Analyze Phase-Locked Loop (PLL) operation for FM demodulation
- Compare performance of different FSK demodulation methods

### Textbook Reference
Haykin & Moher, Chapter 7.3–7.4

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from scipy.special import erfc
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: Coherent FSK Detection

**Coherent detection** uses matched filters (correlators) tuned to each FSK tone.
The receiver multiplies the received signal by local copies of each tone and
integrates over one bit period.

**Block diagram:**
```
                  ┌─[× cos(2πf₀t)]─[∫dt over T_b]─[Sample]─┐
Received r(t) ────┤                                         ├─[Compare]─→ Decision
                  └─[× cos(2πf₁t)]─[∫dt over T_b]─[Sample]─┘
```

**Decision rule:** Choose the frequency whose correlator output is larger.

**Orthogonality requirement:** For optimal detection, the minimum frequency
separation is $\Delta f = R_b / 2 = 1/(2T_b)$ (MSK condition).

In [ ]:
# === Part 1: Coherent FSK Detection with Correlators ===

# FSK parameters
Rb = 1000          # Bit rate (bps)
Tb = 1 / Rb        # Bit period (s)
fc = 5000           # Carrier frequency (Hz)
freq_sep = Rb       # Frequency separation = Rb (orthogonal, h=1)
f0 = fc - freq_sep / 2   # Frequency for bit '0'
f1 = fc + freq_sep / 2   # Frequency for bit '1'
A = 1.0             # Amplitude
fs = 80000          # Sampling rate

# Generate a short bit sequence
bits = np.array([1, 0, 1, 1, 0, 0, 1, 0])
N_bits = len(bits)
samples_per_bit = int(Tb * fs)

# Generate FSK signal (CPFSK)
t_total = np.arange(N_bits * samples_per_bit) / fs
fsk_signal = np.zeros(len(t_total))
for i, bit in enumerate(bits):
    idx_start = i * samples_per_bit
    idx_end = (i + 1) * samples_per_bit
    t_bit = t_total[idx_start:idx_end]
    f_inst = f1 if bit == 1 else f0
    fsk_signal[idx_start:idx_end] = A * np.cos(2 * np.pi * f_inst * t_bit)

# Add AWGN noise
SNR_dB = 10
noise_power = (A**2 / 2) / (10**(SNR_dB / 10))
noise = np.sqrt(noise_power) * np.random.randn(len(t_total))
received = fsk_signal + noise

# Coherent detection: correlate with each tone
detected_bits = np.zeros(N_bits, dtype=int)
corr0_vals = np.zeros(N_bits)
corr1_vals = np.zeros(N_bits)

for i in range(N_bits):
    idx_start = i * samples_per_bit
    idx_end = (i + 1) * samples_per_bit
    t_bit = t_total[idx_start:idx_end]
    r_bit = received[idx_start:idx_end]
    
    # Correlator outputs (multiply and integrate)
    corr0 = np.trapezoid(r_bit * np.cos(2 * np.pi * f0 * t_bit), t_bit)
    corr1 = np.trapezoid(r_bit * np.cos(2 * np.pi * f1 * t_bit), t_bit)
    corr0_vals[i] = corr0
    corr1_vals[i] = corr1
    detected_bits[i] = 1 if corr1 > corr0 else 0

# Plot results
fig, axes = plt.subplots(3, 1, figsize=(12, 8), gridspec_kw={'height_ratios': [1, 2, 1.5]})

# Transmitted bits
for i, bit in enumerate(bits):
    axes[0].fill_between([i*Tb*1000, (i+1)*Tb*1000], bit, alpha=0.4,
                         color='C0' if bit == 1 else 'C1')
    axes[0].text((i+0.5)*Tb*1000, 0.5, str(bit), ha='center', va='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Bit Value')
axes[0].set_title('Coherent FSK Detection', fontsize=14, fontweight='bold')
axes[0].set_ylim(-0.1, 1.1)
axes[0].set_xlim(0, N_bits*Tb*1000)

# Received signal
axes[1].plot(t_total * 1000, received, alpha=0.7, linewidth=0.5)
for i in range(1, N_bits):
    axes[1].axvline(i * Tb * 1000, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Amplitude')
axes[1].set_xlim(0, N_bits*Tb*1000)

# Correlator outputs
bit_centers = (np.arange(N_bits) + 0.5) * Tb * 1000
width = Tb * 1000 * 0.35
axes[2].bar(bit_centers - width/2, corr0_vals * 1000, width, label=f'Corr f\u2080 = {f0:.0f} Hz (bit 0)', alpha=0.8)
axes[2].bar(bit_centers + width/2, corr1_vals * 1000, width, label=f'Corr f\u2081 = {f1:.0f} Hz (bit 1)', alpha=0.8)
axes[2].axhline(0, color='black', linewidth=0.5)
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Correlator Output (\u00d710\u207b\u00b3)')
axes[2].legend(loc='upper right', fontsize=9)
axes[2].set_xlim(0, N_bits*Tb*1000)

plt.tight_layout()
plt.show()

errors = np.sum(bits != detected_bits)
print(f"FSK Parameters: f\u2080 = {f0:.0f} Hz, f\u2081 = {f1:.0f} Hz, Rb = {Rb} bps")
print(f"Frequency separation: {freq_sep} Hz = {freq_sep/Rb:.1f} \u00d7 Rb (h = {freq_sep/Rb:.1f})")
print(f"\nTransmitted: {bits}")
print(f"Detected:    {detected_bits}")
print(f"Bit errors:  {errors}/{N_bits} at SNR = {SNR_dB} dB")

## Part 2: Non-Coherent FSK Detection

**Non-coherent detection** does not require carrier phase synchronization.
Instead, it uses bandpass filters centered at each tone frequency followed
by envelope detectors.

**Block diagram:**
```
                  ┌─[BPF f₀]─[Envelope Detector]─[Sample]─┐
Received r(t) ────┤                                         ├─[Compare]─→ Decision
                  └─[BPF f₁]─[Envelope Detector]─[Sample]─┘
```

**Key advantage:** No need for a phase-locked local oscillator — the envelope
detector extracts signal energy regardless of carrier phase.

**Trade-off:** ~1–3 dB worse BER than coherent detection.

In [ ]:
# === Part 2: Non-Coherent FSK Detection ===

# Design bandpass filters centered at f0 and f1
bw = 1.5 * Rb  # Filter bandwidth (Hz)

# BPF for f0
f0_low = (f0 - bw/2) / (fs/2)
f0_high = (f0 + bw/2) / (fs/2)
b0, a0 = signal.butter(4, [f0_low, f0_high], btype='band')

# BPF for f1
f1_low = (f1 - bw/2) / (fs/2)
f1_high = (f1 + bw/2) / (fs/2)
b1, a1 = signal.butter(4, [f1_low, f1_high], btype='band')

# Filter the received signal
filtered_0 = signal.filtfilt(b0, a0, received)
filtered_1 = signal.filtfilt(b1, a1, received)

# Envelope detection using analytic signal (Hilbert transform)
env_0 = np.abs(signal.hilbert(filtered_0))
env_1 = np.abs(signal.hilbert(filtered_1))

# Decision: sample envelope at end of each bit period
nc_detected = np.zeros(N_bits, dtype=int)
env0_samples = np.zeros(N_bits)
env1_samples = np.zeros(N_bits)

for i in range(N_bits):
    # Average envelope over last quarter of bit period for stability
    idx_start = int((i + 0.75) * samples_per_bit)
    idx_end = (i + 1) * samples_per_bit
    env0_samples[i] = np.mean(env_0[idx_start:idx_end])
    env1_samples[i] = np.mean(env_1[idx_start:idx_end])
    nc_detected[i] = 1 if env1_samples[i] > env0_samples[i] else 0

# Plot non-coherent detection
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# BPF output for f0
axes[0].plot(t_total * 1000, filtered_0, alpha=0.4, linewidth=0.5, color='C1')
axes[0].plot(t_total * 1000, env_0, color='C1', linewidth=2, label='Envelope')
axes[0].set_ylabel('BPF f\u2080 Output')
axes[0].set_title('Non-Coherent FSK Detection: Envelope Outputs', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].set_xlim(0, N_bits*Tb*1000)
for i in range(1, N_bits):
    axes[0].axvline(i * Tb * 1000, color='gray', linestyle='--', alpha=0.4)

# BPF output for f1
axes[1].plot(t_total * 1000, filtered_1, alpha=0.4, linewidth=0.5, color='C0')
axes[1].plot(t_total * 1000, env_1, color='C0', linewidth=2, label='Envelope')
axes[1].set_ylabel('BPF f\u2081 Output')
axes[1].legend(loc='upper right')
axes[1].set_xlim(0, N_bits*Tb*1000)
for i in range(1, N_bits):
    axes[1].axvline(i * Tb * 1000, color='gray', linestyle='--', alpha=0.4)

# Decision comparison
bit_centers = (np.arange(N_bits) + 0.5) * Tb * 1000
width = Tb * 1000 * 0.35
axes[2].bar(bit_centers - width/2, env0_samples, width, label='Env f\u2080 (bit 0)', color='C1', alpha=0.8)
axes[2].bar(bit_centers + width/2, env1_samples, width, label='Env f\u2081 (bit 1)', color='C0', alpha=0.8)
# Mark decisions
for i in range(N_bits):
    marker = '\u2713' if nc_detected[i] == bits[i] else '\u2717'
    axes[2].text(bit_centers[i], max(env0_samples[i], env1_samples[i]) * 1.05,
                 f'{nc_detected[i]} {marker}', ha='center', fontsize=10, fontweight='bold')
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Envelope Amplitude')
axes[2].legend(loc='upper right', fontsize=9)
axes[2].set_xlim(0, N_bits*Tb*1000)

plt.tight_layout()
plt.show()

nc_errors = np.sum(bits != nc_detected)
print(f"Non-Coherent Detection Results:")
print(f"Transmitted: {bits}")
print(f"Detected:    {nc_detected}")
print(f"Bit errors:  {nc_errors}/{N_bits}")
print(f"\nFilter bandwidth: {bw:.0f} Hz per channel")
print(f"Channel spacing:  {f1 - f0:.0f} Hz")

## Part 3: Frequency Discriminator

A **frequency discriminator** converts frequency variations into amplitude
variations — it is the classical method for FM demodulation.

**Operation:**
1. Take the derivative of the FM signal: $\frac{d}{dt}[\cos(\theta(t))] = -\dot{\theta}(t) \sin(\theta(t))$
2. The envelope of the derivative is proportional to the instantaneous frequency
3. An envelope detector recovers the frequency variation (the message)

**For FSK:** The discriminator output is high for $f_1$ and low for $f_0$,
producing a rectangular waveform that directly recovers the bit stream.

In [ ]:
# === Part 3: Frequency Discriminator ===

# Generate a clean FSK signal for discriminator demo
# Use continuous-phase FSK for smoother discriminator output
t_cpfsk = np.arange(N_bits * samples_per_bit) / fs
freq_signal = np.zeros(len(t_cpfsk))
for i, bit in enumerate(bits):
    idx_s = i * samples_per_bit
    idx_e = (i + 1) * samples_per_bit
    freq_signal[idx_s:idx_e] = f1 if bit == 1 else f0

# Generate CPFSK: integrate frequency to get phase
phase = 2 * np.pi * np.cumsum(freq_signal) / fs
cpfsk = A * np.cos(phase)

# Add moderate noise
noise_disc = np.sqrt(noise_power) * np.random.randn(len(t_cpfsk))
cpfsk_noisy = cpfsk + noise_disc

# Discriminator: differentiate and take envelope
# Approximate derivative
derivative = np.diff(cpfsk_noisy) * fs
derivative = np.append(derivative, derivative[-1])  # pad to same length

# Envelope of derivative (instantaneous frequency)
analytic_deriv = signal.hilbert(derivative)
disc_envelope = np.abs(analytic_deriv)

# Lowpass filter to smooth
cutoff = 3 * Rb / (fs / 2)
b_lp, a_lp = signal.butter(3, cutoff, btype='low')
disc_output = signal.filtfilt(b_lp, a_lp, disc_envelope)

# Decision threshold (midpoint between expected levels)
level_0 = 2 * np.pi * f0 * A  # Expected envelope for f0
level_1 = 2 * np.pi * f1 * A  # Expected envelope for f1
threshold = (level_0 + level_1) / 2

# Detect bits
disc_detected = np.zeros(N_bits, dtype=int)
for i in range(N_bits):
    idx_s = int((i + 0.25) * samples_per_bit)
    idx_e = int((i + 0.75) * samples_per_bit)
    disc_detected[i] = 1 if np.mean(disc_output[idx_s:idx_e]) > threshold else 0

# Plot discriminator operation
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

axes[0].plot(t_cpfsk * 1000, cpfsk_noisy, linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Frequency Discriminator for FSK Demodulation', fontsize=14, fontweight='bold')
axes[0].set_xlim(0, N_bits*Tb*1000)
for i in range(1, N_bits):
    axes[0].axvline(i * Tb * 1000, color='gray', linestyle='--', alpha=0.4)

axes[1].plot(t_cpfsk * 1000, disc_output / (2 * np.pi), color='C2', linewidth=1.5)
axes[1].axhline(threshold / (2 * np.pi), color='red', linestyle='--', linewidth=1.5, label='Threshold')
axes[1].axhline(f0, color='C1', linestyle=':', alpha=0.5, label=f'f\u2080 = {f0:.0f} Hz')
axes[1].axhline(f1, color='C0', linestyle=':', alpha=0.5, label=f'f\u2081 = {f1:.0f} Hz')
axes[1].set_ylabel('Frequency (Hz)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_xlim(0, N_bits*Tb*1000)
for i in range(1, N_bits):
    axes[1].axvline(i * Tb * 1000, color='gray', linestyle='--', alpha=0.4)

# Recovered bits
for i in range(N_bits):
    color = 'C0' if disc_detected[i] == 1 else 'C1'
    axes[2].fill_between([i*Tb*1000, (i+1)*Tb*1000], disc_detected[i], alpha=0.4, color=color)
    match = '\u2713' if disc_detected[i] == bits[i] else '\u2717'
    axes[2].text((i+0.5)*Tb*1000, 0.5, f'{disc_detected[i]} {match}',
                 ha='center', va='center', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Detected Bit')
axes[2].set_ylim(-0.1, 1.1)
axes[2].set_xlim(0, N_bits*Tb*1000)

plt.tight_layout()
plt.show()

disc_errors = np.sum(bits != disc_detected)
print(f"Frequency Discriminator Results:")
print(f"Transmitted: {bits}")
print(f"Detected:    {disc_detected}")
print(f"Bit errors:  {disc_errors}/{N_bits}")
print(f"\nExpected levels: f\u2080 \u2192 {level_0/(2*np.pi):.0f} Hz, f\u2081 \u2192 {level_1/(2*np.pi):.0f} Hz")
print(f"Decision threshold: {threshold/(2*np.pi):.0f} Hz")

## Part 4: Phase-Locked Loop (PLL) for FM Demodulation

A **PLL** tracks the instantaneous frequency of the input signal. Its control
voltage is proportional to the frequency deviation — exactly the message signal
for FM.

**PLL components:**
1. **Phase Detector:** Compares input phase to VCO phase
2. **Loop Filter:** Lowpass filter that smooths the error signal
3. **VCO (Voltage-Controlled Oscillator):** Output frequency tracks control voltage

```
Input → [Phase Detector] → [Loop Filter] → Control Voltage (= message)
              ↑                                ↓
              └────────── [VCO] ───────────┘
```

**Key insight:** When the PLL is locked, the VCO frequency matches the input
frequency. The control voltage that drives the VCO therefore tracks the
instantaneous frequency deviation.

In [ ]:
# === Part 4: Simple PLL FM Demodulator Simulation ===

# PLL parameters
Kp = 800.0       # Phase detector gain
Ki = 50000.0      # Integral gain (loop filter)
vco_center = fc   # VCO free-running frequency (Hz)
Kvco = 2000.0     # VCO gain (Hz/V)

# Run PLL on the noisy CPFSK signal
N = len(cpfsk_noisy)
phase_vco = np.zeros(N)
freq_vco = np.zeros(N)
control_voltage = np.zeros(N)
phase_error = np.zeros(N)
integrator = 0.0

for n in range(1, N):
    # Phase detector: multiply input by VCO quadrature output
    # sin(phase_in - phase_vco) \approx phase_in - phase_vco for small error
    pd_out = cpfsk_noisy[n] * (-np.sin(phase_vco[n-1]))
    
    # Loop filter (PI controller)
    integrator += pd_out / fs
    control_voltage[n] = Kp * pd_out + Ki * integrator
    
    # VCO: frequency = center + Kvco * control_voltage
    freq_vco[n] = vco_center + Kvco * control_voltage[n]
    phase_vco[n] = phase_vco[n-1] + 2 * np.pi * freq_vco[n] / fs

# Lowpass filter the control voltage to get smooth frequency estimate
b_lp2, a_lp2 = signal.butter(3, 4 * Rb / (fs / 2), btype='low')
freq_estimate = signal.filtfilt(b_lp2, a_lp2, freq_vco)

# PLL bit detection
pll_threshold = fc
pll_detected = np.zeros(N_bits, dtype=int)
for i in range(N_bits):
    idx_s = int((i + 0.3) * samples_per_bit)
    idx_e = int((i + 0.7) * samples_per_bit)
    pll_detected[i] = 1 if np.mean(freq_estimate[idx_s:idx_e]) > pll_threshold else 0

# Plot PLL operation
fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Input frequency (true)
axes[0].plot(t_cpfsk * 1000, freq_signal, 'k-', linewidth=1.5, label='True frequency')
axes[0].axhline(fc, color='gray', linestyle=':', alpha=0.5)
axes[0].set_ylabel('Frequency (Hz)')
axes[0].set_title('PLL FM Demodulator: Tracking FSK Signal', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].set_xlim(0, N_bits*Tb*1000)

# VCO frequency tracking
axes[1].plot(t_cpfsk * 1000, freq_signal, 'k--', linewidth=1, alpha=0.4, label='True')
axes[1].plot(t_cpfsk * 1000, freq_estimate, 'C3', linewidth=1.5, label='PLL estimate')
axes[1].axhline(fc, color='gray', linestyle=':', alpha=0.5, label=f'f_c = {fc} Hz')
axes[1].set_ylabel('VCO Frequency (Hz)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_xlim(0, N_bits*Tb*1000)
for i in range(1, N_bits):
    axes[1].axvline(i * Tb * 1000, color='gray', linestyle='--', alpha=0.3)

# Detected bits
for i in range(N_bits):
    color = 'C0' if pll_detected[i] == 1 else 'C1'
    axes[2].fill_between([i*Tb*1000, (i+1)*Tb*1000], pll_detected[i], alpha=0.4, color=color)
    match = '\u2713' if pll_detected[i] == bits[i] else '\u2717'
    axes[2].text((i+0.5)*Tb*1000, 0.5, f'{pll_detected[i]} {match}',
                 ha='center', va='center', fontsize=11, fontweight='bold')
axes[2].set_xlabel('Time (ms)')
axes[2].set_ylabel('Detected Bit')
axes[2].set_ylim(-0.1, 1.1)
axes[2].set_xlim(0, N_bits*Tb*1000)

plt.tight_layout()
plt.show()

pll_errors = np.sum(bits != pll_detected)
print(f"PLL Demodulator Results:")
print(f"Transmitted: {bits}")
print(f"Detected:    {pll_detected}")
print(f"Bit errors:  {pll_errors}/{N_bits}")
print(f"\nPLL parameters: Kp = {Kp}, Ki = {Ki}, Kvco = {Kvco} Hz/V")
print(f"VCO center frequency: {vco_center} Hz")

## Part 5: BER Performance Comparison

The theoretical bit error rate (BER) for binary FSK:

| Method | BER Formula | Notes |
|--------|-------------|-------|
| **Coherent FSK** | $P_e = Q\left(\sqrt{E_b/N_0}\right)$ | Optimal, requires carrier sync |
| **Non-coherent FSK** | $P_e = \frac{1}{2} e^{-E_b/(2N_0)}$ | Simpler, ~1 dB penalty |
| **Coherent BPSK** (reference) | $P_e = Q\left(\sqrt{2E_b/N_0}\right)$ | 3 dB better than coherent FSK |

where $Q(x) = \frac{1}{2}\text{erfc}(x/\sqrt{2})$.

In [ ]:
# === Part 5: BER Curves — Coherent vs Non-Coherent FSK ===

EbN0_dB = np.linspace(0, 16, 200)
EbN0 = 10**(EbN0_dB / 10)

# Theoretical BER formulas
# Q(x) = 0.5 * erfc(x / sqrt(2))
BER_coherent_fsk = 0.5 * erfc(np.sqrt(EbN0 / 2))       # Q(sqrt(Eb/N0))
BER_noncoherent_fsk = 0.5 * np.exp(-EbN0 / 2)           # (1/2) exp(-Eb/2N0)
BER_bpsk = 0.5 * erfc(np.sqrt(EbN0))                    # Q(sqrt(2Eb/N0))

fig, ax = plt.subplots(figsize=(10, 7))

ax.semilogy(EbN0_dB, BER_bpsk, 'b-', linewidth=2.5, label='Coherent BPSK (reference)')
ax.semilogy(EbN0_dB, BER_coherent_fsk, 'g-', linewidth=2.5, label='Coherent FSK')
ax.semilogy(EbN0_dB, BER_noncoherent_fsk, 'r--', linewidth=2.5, label='Non-coherent FSK')

# Mark key BER levels
for target_ber, label in [(1e-3, '10\u207b\u00b3'), (1e-5, '10\u207b\u2075')]:
    ax.axhline(target_ber, color='gray', linestyle=':', alpha=0.4)
    ax.text(0.3, target_ber * 1.3, f'BER = {label}', fontsize=9, color='gray')

# Annotate the SNR penalty
# Find Eb/N0 for BER = 1e-4 for each method
target = 1e-4
idx_coh = np.argmin(np.abs(BER_coherent_fsk - target))
idx_nc = np.argmin(np.abs(BER_noncoherent_fsk - target))
idx_bpsk = np.argmin(np.abs(BER_bpsk - target))

penalty_nc = EbN0_dB[idx_nc] - EbN0_dB[idx_coh]
penalty_fsk = EbN0_dB[idx_coh] - EbN0_dB[idx_bpsk]

ax.annotate('', xy=(EbN0_dB[idx_coh], target), xytext=(EbN0_dB[idx_nc], target),
            arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
ax.text((EbN0_dB[idx_coh] + EbN0_dB[idx_nc])/2, target * 3,
        f'~{penalty_nc:.1f} dB\npenalty', ha='center', fontsize=10, color='red')

ax.annotate('', xy=(EbN0_dB[idx_bpsk], target*0.3), xytext=(EbN0_dB[idx_coh], target*0.3),
            arrowprops=dict(arrowstyle='<->', color='blue', lw=1.5))
ax.text((EbN0_dB[idx_bpsk] + EbN0_dB[idx_coh])/2, target * 0.1,
        f'~{penalty_fsk:.1f} dB\n(BPSK advantage)', ha='center', fontsize=10, color='blue')

ax.set_xlabel('$E_b/N_0$ (dB)', fontsize=13)
ax.set_ylabel('Bit Error Rate (BER)', fontsize=13)
ax.set_title('BER Comparison: FSK Demodulation Methods', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower left')
ax.set_ylim(1e-7, 0.5)
ax.set_xlim(0, 16)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# Print comparison table
print("BER Performance Comparison at BER = 10\u207b\u2074:")
print(f"{'Method':<25} {'Eb/N0 Required':>15} {'Penalty vs BPSK':>17}")
print("-" * 60)
print(f"{'Coherent BPSK':<25} {EbN0_dB[idx_bpsk]:>12.1f} dB {'(reference)':>17}")
print(f"{'Coherent FSK':<25} {EbN0_dB[idx_coh]:>12.1f} dB {penalty_fsk:>14.1f} dB")
print(f"{'Non-coherent FSK':<25} {EbN0_dB[idx_nc]:>12.1f} dB {penalty_fsk + penalty_nc:>14.1f} dB")
print(f"\nNon-coherent penalty vs coherent FSK: ~{penalty_nc:.1f} dB")
print(f"FSK penalty vs BPSK: ~{penalty_fsk:.1f} dB (FSK uses wider bandwidth)")

## Part 6: Practical FSK Applications

FSK remains widely used in systems where **simplicity**, **constant envelope**,
and **robustness** matter more than spectral efficiency.

| Application | Standard | Data Rate | Frequencies | Modulation |
|-------------|----------|-----------|-------------|------------|
| Telephone modem | Bell 103 | 300 bps | 1070/1270 Hz | Binary FSK |
| Caller ID | Bell 202 | 1200 bps | 1200/2200 Hz | Binary FSK |
| APRS (Ham radio) | AX.25 | 1200 bps | 1200/2200 Hz | AFSK on VHF |
| Bluetooth | IEEE 802.15 | 1 Mbps | \u00b1160 kHz | GFSK |
| LoRa (IoT) | Semtech | 0.3–50 kbps | Chirp sweep | CSS (related to FSK) |

In [ ]:
# === Part 6: Bell 103 Modem — Classic FSK Application ===

# Bell 103 modem parameters (300 bps full-duplex)
Rb_modem = 300        # bps
Tb_modem = 1 / Rb_modem
fs_modem = 16000      # Sampling rate

# Originate channel
f0_orig = 1070   # Space (bit 0)
f1_orig = 1270   # Mark (bit 1)

# Answer channel
f0_ans = 2025    # Space (bit 0)
f1_ans = 2225    # Mark (bit 1)

# Generate sample data: "Hi" in ASCII
text = "Hi"
ascii_bits = []
for ch in text:
    byte = format(ord(ch), '08b')
    ascii_bits.extend([int(b) for b in byte])

N_modem = len(ascii_bits)
samples_per_bit_modem = int(Tb_modem * fs_modem)
t_modem = np.arange(N_modem * samples_per_bit_modem) / fs_modem

# Generate originate channel FSK
orig_signal = np.zeros(len(t_modem))
freq_orig = np.zeros(len(t_modem))
for i, bit in enumerate(ascii_bits):
    idx_s = i * samples_per_bit_modem
    idx_e = (i + 1) * samples_per_bit_modem
    f = f1_orig if bit == 1 else f0_orig
    freq_orig[idx_s:idx_e] = f

phase_orig = 2 * np.pi * np.cumsum(freq_orig) / fs_modem
orig_signal = 0.5 * np.cos(phase_orig)

# Plot Bell 103 modem signal
fig, axes = plt.subplots(3, 1, figsize=(14, 7))

# Bit sequence
for i, bit in enumerate(ascii_bits):
    color = 'C0' if bit == 1 else 'C1'
    axes[0].fill_between([i*Tb_modem*1000, (i+1)*Tb_modem*1000], bit, alpha=0.3, color=color)
# Mark byte boundaries
for i in range(0, N_modem + 1, 8):
    axes[0].axvline(i * Tb_modem * 1000, color='black', linewidth=1.5, alpha=0.6)
axes[0].text(4 * Tb_modem * 1000, 1.15, f"'{text[0]}' (0x{ord(text[0]):02X})", ha='center', fontsize=11)
axes[0].text(12 * Tb_modem * 1000, 1.15, f"'{text[1]}' (0x{ord(text[1]):02X})", ha='center', fontsize=11)
axes[0].set_ylabel('Bit Value')
axes[0].set_title('Bell 103 Modem: "Hi" at 300 bps FSK', fontsize=14, fontweight='bold')
axes[0].set_ylim(-0.1, 1.4)
axes[0].set_xlim(0, N_modem * Tb_modem * 1000)

# Time-domain signal (first few bits)
show_bits = 6
show_samples = show_bits * samples_per_bit_modem
axes[1].plot(t_modem[:show_samples] * 1000, orig_signal[:show_samples], linewidth=0.8)
for i in range(1, show_bits):
    axes[1].axvline(i * Tb_modem * 1000, color='gray', linestyle='--', alpha=0.4)
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (ms)')
axes[1].set_xlim(0, show_bits * Tb_modem * 1000)

# Spectrum
N_fft = len(orig_signal)
freqs = fftfreq(N_fft, 1/fs_modem)[:N_fft//2]
spectrum = np.abs(fft(orig_signal))[:N_fft//2]
spectrum_dB = 20 * np.log10(spectrum / np.max(spectrum) + 1e-10)

axes[2].plot(freqs, spectrum_dB, linewidth=1)
axes[2].axvline(f0_orig, color='C1', linestyle='--', alpha=0.7, label=f'f\u2080 = {f0_orig} Hz')
axes[2].axvline(f1_orig, color='C0', linestyle='--', alpha=0.7, label=f'f\u2081 = {f1_orig} Hz')
# Show answer channel region
axes[2].axvspan(f0_ans - 200, f1_ans + 200, alpha=0.1, color='green', label='Answer channel')
axes[2].set_xlabel('Frequency (Hz)')
axes[2].set_ylabel('Magnitude (dB)')
axes[2].set_xlim(0, 3500)
axes[2].set_ylim(-60, 5)
axes[2].legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print("Bell 103 Modem Specifications:")
print(f"  Data rate:        {Rb_modem} bps (full-duplex)")
print(f"  Originate:        f\u2080 = {f0_orig} Hz (space), f\u2081 = {f1_orig} Hz (mark)")
print(f"  Answer:           f\u2080 = {f0_ans} Hz (space), f\u2081 = {f1_ans} Hz (mark)")
print(f"  Tone spacing:     {f1_orig - f0_orig} Hz (originate), {f1_ans - f0_ans} Hz (answer)")
print(f"  Full-duplex via FDM: two channels share one phone line")
print(f"\nASCII encoding of '{text}':")
for i, ch in enumerate(text):
    byte_bits = ascii_bits[i*8:(i+1)*8]
    print(f"  '{ch}' = 0x{ord(ch):02X} = {''.join(map(str, byte_bits))}")

## Summary

### FSK Demodulation Methods Comparison

| Method | Complexity | BER Performance | Carrier Sync? | Key Application |
|--------|-----------|-----------------|---------------|------------------|
| **Coherent (correlator)** | High | Best: $Q(\sqrt{E_b/N_0})$ | Yes | High-performance links |
| **Non-coherent (envelope)** | Low | ~1 dB worse: $\frac{1}{2}e^{-E_b/2N_0}$ | No | Low-cost receivers |
| **Discriminator** | Medium | Good for high SNR | No | Classic FM receivers |
| **PLL** | Medium | Good, tracks drift | Implicit | Modern SDR, ICs |

### Key Formulas

| Quantity | Formula |
|----------|--------|
| Min. orthogonal spacing | $\Delta f_{\min} = R_b / 2$ |
| Coherent FSK BER | $P_e = Q\left(\sqrt{E_b/N_0}\right)$ |
| Non-coherent FSK BER | $P_e = \frac{1}{2} \exp\left(-\frac{E_b}{2N_0}\right)$ |
| BPSK BER (reference) | $P_e = Q\left(\sqrt{2E_b/N_0}\right)$ |
| FSK bandwidth | $B \approx 2(\Delta f + R_b)$ |

### Key Takeaways

1. **Coherent vs non-coherent** is a complexity-vs-performance trade-off
2. FSK requires ~3 dB more $E_b/N_0$ than BPSK for the same BER
3. FSK uses more bandwidth than PSK, but has **constant envelope** (power-efficient)
4. PLLs provide automatic frequency tracking and are standard in modern receivers
5. FSK remains relevant in IoT (LoRa), Bluetooth (GFSK), and amateur radio (APRS)

### Next Topics
- **Lesson 13:** FM Generation & Demodulation / BPSK Theory
- Reading Quiz 5 at beginning of class
- Homework 3 due